# Évaluation V5 depuis checkpoint-4800

Validation loss: 2.178 (meilleur checkpoint)

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os
import json
import re
import random
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter

from unsloth import FastVisionModel
from peft import PeftModel
from transformers import AutoProcessor

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.9.0+cu128
CUDA: True


In [2]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
CHECKPOINT_PATH = BASE_DIR / "checkpoints" / "stage2_v5_unified" / "final_model"
VAL_DATA_PATH = BASE_DIR / "data" / "stage1" / "cassava_val.json"

MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]

# Stage 1 results for comparison
STAGE1_RESULTS = {
    "global": 80.52,
    "CBB": 55.13,
    "CBSD": 83.99,
    "CGM": 56.47,
    "CMD": 85.98,
    "Healthy": 75.37
}

print(f"Checkpoint: {CHECKPOINT_PATH}")

Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified/final_model


In [3]:
# ============================================================
# CELL 3: CHARGER MODÈLE DEPUIS CHECKPOINT
# ============================================================
print("Chargement du modèle de base...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
)

print(f"Chargement du checkpoint V5: {CHECKPOINT_PATH}")
model = PeftModel.from_pretrained(model, str(CHECKPOINT_PATH))

processor = AutoProcessor.from_pretrained(str(CHECKPOINT_PATH))

# Mode inférence
FastVisionModel.for_inference(model)
print("✅ Modèle chargé en mode inférence")

Chargement du modèle de base...
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Chargement du checkpoint V5: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified/final_model
✅ Modèle chargé en mode inférence


In [4]:
# ============================================================
# CELL 4: CHARGER DONNÉES DE VALIDATION
# ============================================================
with open(VAL_DATA_PATH, 'r', encoding='utf-8') as f:
    val_data = json.load(f)

print(f"Validation samples: {len(val_data)}")

# Distribution
def get_class_from_response(response: str) -> str:
    response_lower = response.lower()
    if "cbb" in response_lower or "bacterial blight" in response_lower:
        return "CBB"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cgm" in response_lower or "green mottle" in response_lower or "green mite" in response_lower:
        return "CGM"
    elif "cmd" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "healthy" in response_lower:
        return "Healthy"
    return "Unknown"

val_dist = Counter()
for s in val_data:
    for turn in s['conversations']:
        if turn['from'] == 'assistant':
            cls = get_class_from_response(turn['value'])
            val_dist[cls] += 1
            break

print("\nDistribution validation:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {val_dist[cls]}")

Validation samples: 3612

Distribution validation:
  CBB: 154
  CBSD: 662
  CGM: 315
  CMD: 2080
  Healthy: 401


In [5]:
# ============================================================
# CELL 5: FONCTIONS D'ÉVALUATION
# ============================================================
def generate_response(image_path: str, question: str, max_new_tokens: int = 256):
    """Génère une réponse pour une image."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Erreur"

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]
    }]

    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[input_text], images=[[image]],
        return_tensors="pt", padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    return response

def extract_disease(response: str) -> str:
    """Extrait la maladie de la réponse."""
    response_lower = response.lower()
    
    # Chercher "diagnostic :" d'abord (format V5)
    if "diagnostic" in response_lower:
        match = re.search(r'diagnostic\s*:\s*([\w\s()]+?)[\.\,]', response_lower)
        if match:
            diag = match.group(1).strip()
            if "cmd" in diag or "mosaic" in diag:
                return "CMD"
            elif "cbsd" in diag or "brown streak" in diag:
                return "CBSD"
            elif "cbb" in diag or "bacterial" in diag:
                return "CBB"
            elif "cgm" in diag or "green" in diag:
                return "CGM"
            elif "healthy" in diag or "sain" in diag:
                return "Healthy"
    
    # Chercher "disease:" (format Stage 1)
    if "disease:" in response_lower:
        match = re.search(r'disease:\s*([\w\s()]+)', response_lower)
        if match:
            diag = match.group(1).strip()
            if "cmd" in diag or "mosaic" in diag:
                return "CMD"
            elif "cbsd" in diag or "brown streak" in diag:
                return "CBSD"
            elif "cbb" in diag or "bacterial" in diag:
                return "CBB"
            elif "cgm" in diag or "green" in diag:
                return "CGM"
            elif "healthy" in diag:
                return "Healthy"
    
    # Fallback - chercher les mots-clés
    if "cmd" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bacterial" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "green mottle" in response_lower or "green mite" in response_lower:
        return "CGM"
    elif "healthy" in response_lower or "sain" in response_lower:
        return "Healthy"
    
    return "Unknown"

print("✅ Fonctions d'évaluation prêtes")

✅ Fonctions d'évaluation prêtes


In [6]:
# ============================================================
# CELL 6: ÉVALUATION CLASSIFICATION
# ============================================================
print("="*60)
print("ÉVALUATION V5 - CLASSIFICATION")
print("="*60)

# Préparer les samples de validation
eval_samples = []
for s in val_data:
    image_path = None
    actual_class = None
    
    for turn in s['conversations']:
        if turn['from'] == 'user':
            match = re.search(r'<img>(.*?)</img>', turn['value'])
            if match:
                image_path = match.group(1)
        elif turn['from'] == 'assistant':
            actual_class = get_class_from_response(turn['value'])
    
    if image_path and os.path.exists(image_path) and actual_class != "Unknown":
        eval_samples.append({
            'image_path': image_path,
            'actual': actual_class
        })

print(f"Samples valides: {len(eval_samples)}")

# Échantillonner 200 samples
random.seed(42)
test_samples = random.sample(eval_samples, min(200, len(eval_samples)))

# Questions
questions = [
    "What cassava disease is shown in this image?",
    "Quelle maladie affecte cette feuille de manioc ?"
]

print(f"\n📊 Évaluation sur {len(eval_samples)} samples...")

results = []
for sample in tqdm(eval_samples, desc="Evaluating"):
    question = random.choice(questions)
    response = generate_response(sample['image_path'], question)
    predicted = extract_disease(response)
    
    results.append({
        'actual': sample['actual'],
        'predicted': predicted,
        'correct': predicted == sample['actual'],
        'response': response[:300]
    })

# Calculer accuracy
correct = sum(1 for r in results if r['correct'])
accuracy = correct / len(results) * 100

print(f"\n" + "="*60)
print(f"RÉSULTATS V5 vs Stage 1")
print(f"="*60)
print(f"\n📊 Accuracy globale:")
print(f"  - Stage 1: {STAGE1_RESULTS['global']:.1f}%")
print(f"  - V5: {accuracy:.1f}%")
diff = accuracy - STAGE1_RESULTS['global']
print(f"  - Différence: {diff:+.1f}%")

# Par classe
print(f"\n📈 Par classe:")
class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        cls_acc = cls_correct / len(cls_samples) * 100
        class_metrics[cls] = cls_acc
        stage1_acc = STAGE1_RESULTS.get(cls, 0)
        cls_diff = cls_acc - stage1_acc
        status = "✅" if cls_diff >= -10 else "⚠️" if cls_diff >= -20 else "❌"
        print(f"  {status} {cls}: {cls_acc:.1f}% (Stage 1: {stage1_acc:.1f}%, Δ{cls_diff:+.1f}%)")

ÉVALUATION V5 - CLASSIFICATION
Samples valides: 3612

📊 Évaluation sur 3612 samples...


Evaluating:   0%|          | 0/3612 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 3612/3612 [1:56:59<00:00,  1.94s/it]  


RÉSULTATS V5 vs Stage 1

📊 Accuracy globale:
  - Stage 1: 80.5%
  - V5: 83.2%
  - Différence: +2.7%

📈 Par classe:
  ✅ CBB: 72.1% (Stage 1: 55.1%, Δ+16.9%)
  ✅ CBSD: 84.3% (Stage 1: 84.0%, Δ+0.3%)
  ✅ CGM: 67.3% (Stage 1: 56.5%, Δ+10.8%)
  ✅ CMD: 95.4% (Stage 1: 86.0%, Δ+9.4%)
  ❌ Healthy: 34.9% (Stage 1: 75.4%, Δ-40.5%)


In [7]:
# ============================================================
# CELL 7: DIAGNOSTIC - RÉPONSES BRUTES
# ============================================================
print("="*60)
print("DIAGNOSTIC: RÉPONSES BRUTES")
print("="*60)

# 10 samples pour diagnostic
diagnostic_samples = random.sample(results, min(10, len(results)))

for i, sample in enumerate(diagnostic_samples):
    print(f"\n{'='*60}")
    print(f"[{i+1}] Classe réelle: {sample['actual']}")
    print(f"    Classe prédite: {sample['predicted']}")
    print(f"    Correct: {'✅' if sample['correct'] else '❌'}")
    print(f"{'='*60}")
    print(f"RÉPONSE:")
    print(f"{sample['response']}")
    if len(sample['response']) >= 300:
        print("...")

DIAGNOSTIC: RÉPONSES BRUTES

[1] Classe réelle: CBSD
    Classe prédite: CBSD
    Correct: ✅
RÉPONSE:
diagnostic : **cbsd** - striure brune du manioc. sévérité : légère/précoce. symptômes observés : taches chlorotiques dispersées sur le limbe, affecte les feuilles matures (partie inférieure), pas les jeunes, racines souvent sans symptômes (infection latente).

[2] Classe réelle: CMD
    Classe prédite: CMD
    Correct: ✅
RÉPONSE:
diagnostic : cassava mosaic disease (cmd). cette feuille de manioc présente des symptômes caractéristiques de cette condition.

[3] Classe réelle: CMD
    Classe prédite: CMD
    Correct: ✅
RÉPONSE:
cette feuille présente des symptômes de **mosaïque du manioc (cmd)**. niveau de sévérité : général (sévérité non spécifiée). les signes visibles incluent : déformation possible des folioles, motifs mosaïque jaune-vert sur les feuilles, nanisme dans les cas avancés.

[4] Classe réelle: CBB
    Classe prédite: CBSD
    Correct: ❌
RÉPONSE:
diagnostic : **cbsd** - stri

In [8]:
# ============================================================
# CELL 8: TEST VQA
# ============================================================
print("="*60)
print("TEST VQA - Questions variées")
print("="*60)

vqa_questions = [
    "Quels sont les signes de maladie que vous observez ?",
    "Que dois-je faire pour traiter cette maladie ?",
    "Est-ce un problème de maladie ou d'insectes ?",
    "Qu'est-ce qui cause ces symptômes ?"
]

vqa_samples = random.sample(eval_samples, min(10, len(eval_samples)))

for i, sample in enumerate(vqa_samples):
    question = random.choice(vqa_questions)
    response = generate_response(sample['image_path'], question)
    
    print(f"\n{'='*60}")
    print(f"[{i+1}] Classe: {sample['actual']}")
    print(f"    Question: {question}")
    print(f"    Réponse: {response[:400]}")
    if len(response) > 400:
        print("...")

TEST VQA - Questions variées



[1] Classe: CBSD
    Question: Est-ce un problème de maladie ou d'insectes ?
    Réponse: cette feuille de manioc est **saine**. elle présente les caractéristiques normales d'un plant en bonne santé : surface lisse et glabre, consistance souple mais ferme, couleur vert foncé à vert moyen uniforme. continuez à bien entretenir votre culture avec un arrosage régulier et une fertilisation adaptée.

[2] Classe: CMD
    Question: Quels sont les signes de maladie que vous observez ?
    Réponse: cette image montre : motifs mosaïque jaune-vert sur les feuilles, déformation possible des folioles, nanisme dans les cas avancés. ces signes sont typiques de la mosaïque du manioc au stade général (sévérité non spécifiée).

[3] Classe: CBSD
    Question: Est-ce un problème de maladie ou d'insectes ?
    Réponse: il s'agit de **striure brune du manioc (cbsd)**, causée par ipomovirus (cbsv, ucbsv). les symptômes distinctifs sont : stries brunes possibles sur tiges, chlorose sur feuilles matures (pas l

In [9]:
# ============================================================
# CELL 9: CONCLUSION
# ============================================================
print("\n" + "="*60)
print("CONCLUSION V5")
print("="*60)

retention = accuracy - STAGE1_RESULTS['global']

if retention >= -5:
    print(f"\n🎉 Rétention excellente! ({retention:+.1f}% vs Stage 1)")
elif retention >= -15:
    print(f"\n✅ Rétention acceptable ({retention:+.1f}% vs Stage 1)")
elif retention >= -25:
    print(f"\n⚠️ Rétention modérée ({retention:+.1f}% vs Stage 1)")
else:
    print(f"\n❌ Rétention insuffisante ({retention:+.1f}% vs Stage 1)")

# Confusion matrix simplifiée
print("\n📊 Matrice de confusion simplifiée:")
confusion = {}
for cls in CLASS_NAMES:
    confusion[cls] = Counter(r['predicted'] for r in results if r['actual'] == cls)

for actual in CLASS_NAMES:
    predictions = confusion[actual]
    if predictions:
        main_pred = predictions.most_common(1)[0]
        print(f"  {actual} → {main_pred[0]} ({main_pred[1]}x)")

print(f"\n📁 Checkpoint évalué: {CHECKPOINT_PATH}")


CONCLUSION V5

🎉 Rétention excellente! (+2.7% vs Stage 1)

📊 Matrice de confusion simplifiée:
  CBB → CBB (111x)
  CBSD → CBSD (558x)
  CGM → CGM (212x)
  CMD → CMD (1984x)
  Healthy → Healthy (140x)

📁 Checkpoint évalué: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified/final_model


In [10]:
# ============================================================
# CELL 10: INVESTIGATION - HEALTHY CLASS CONFUSION
# ============================================================
print("="*60)
print("INVESTIGATION: HEALTHY CLASS CONFUSION")
print("="*60)

# Filtrer les samples Healthy
healthy_samples = [r for r in results if r['actual'] == 'Healthy']
print(f"\nSamples Healthy dans l'évaluation: {len(healthy_samples)}")

# Distribution des prédictions pour Healthy
healthy_predictions = Counter(r['predicted'] for r in healthy_samples)
print("\n📊 Quand la classe réelle est Healthy, le modèle prédit:")
for pred, count in healthy_predictions.most_common():
    pct = count / len(healthy_samples) * 100
    status = "✅" if pred == "Healthy" else "❌"
    print(f"  {status} {pred}: {count} ({pct:.1f}%)")

# Matrice de confusion complète
print("\n📊 Matrice de confusion complète:")
print(f"{'Actual':<10} | " + " | ".join(f"{c:<8}" for c in CLASS_NAMES + ["Unknown"]))
print("-" * 75)
for actual in CLASS_NAMES:
    row_samples = [r for r in results if r['actual'] == actual]
    if row_samples:
        counts = []
        for pred in CLASS_NAMES + ["Unknown"]:
            count = sum(1 for r in row_samples if r['predicted'] == pred)
            counts.append(f"{count:<8}")
        print(f"{actual:<10} | " + " | ".join(counts))

# Exemples de Healthy mal classifiés
print("\n" + "="*60)
print("EXEMPLES: Healthy → Maladie (erreurs)")
print("="*60)

healthy_errors = [r for r in healthy_samples if not r['correct']]
for i, sample in enumerate(healthy_errors[:5]):
    print(f"\n[{i+1}] Healthy → {sample['predicted']}")
    print(f"    Réponse: {sample['response'][:250]}")
    if len(sample['response']) > 250:
        print("...")

# Analyse des patterns d'erreur
print("\n" + "="*60)
print("ANALYSE DES PATTERNS D'ERREUR")
print("="*60)

# Quelles maladies "volent" les samples Healthy?
if healthy_errors:
    error_dist = Counter(r['predicted'] for r in healthy_errors)
    print("\nLes samples Healthy sont principalement confondus avec:")
    for disease, count in error_dist.most_common():
        pct = count / len(healthy_errors) * 100
        print(f"  → {disease}: {count} erreurs ({pct:.1f}%)")

    # Recommandation
    top_confusion = error_dist.most_common(1)[0][0]
    print(f"\n💡 Recommandation: Le modèle confond Healthy avec {top_confusion}.")
    print("   Causes possibles:")
    print("   - Sous-représentation de Healthy dans les données d'entraînement")
    print("   - Biais vers la détection de maladies (format 'Diagnostic:')")
    print("   - Similarité visuelle entre feuilles saines et certains stades précoces")

INVESTIGATION: HEALTHY CLASS CONFUSION

Samples Healthy dans l'évaluation: 401

📊 Quand la classe réelle est Healthy, le modèle prédit:
  ✅ Healthy: 140 (34.9%)
  ❌ CBSD: 94 (23.4%)
  ❌ Unknown: 72 (18.0%)
  ❌ CBB: 50 (12.5%)
  ❌ CGM: 26 (6.5%)
  ❌ CMD: 19 (4.7%)

📊 Matrice de confusion complète:
Actual     | CBB      | CBSD     | CGM      | CMD      | Healthy  | Unknown 
---------------------------------------------------------------------------
CBB        | 111      | 23       | 3        | 5        | 11       | 1       
CBSD       | 43       | 558      | 11       | 36       | 8        | 6       
CGM        | 14       | 24       | 212      | 56       | 5        | 4       
CMD        | 13       | 40       | 33       | 1984     | 7        | 3       
Healthy    | 50       | 94       | 26       | 19       | 140      | 72      

EXEMPLES: Healthy → Maladie (erreurs)

[1] Healthy → CGM
    Réponse: diagnostic : cassava green mottle (cgm). cette feuille de manioc présente des symptômes carac